In [18]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import joblib

In [2]:
df=pd.read_csv(r"C:\Users\DELL\Desktop\Price_Pulse\dynamic_pricing.csv")

In [3]:
df['demand_supply_ratio']=df['Number_of_Riders']/(df['Number_of_Drivers']+1)
df['high_demand_flag']=(df['Number_of_Riders'] > df['Number_of_Riders'].quantile(0.75)).astype(int)
df['low_supply_flag']=(df['Number_of_Drivers'] < df['Number_of_Drivers'].quantile(0.25)).astype(int) 
df['surge_indicator']=((df['high_demand_flag']==1) & (df['low_supply_flag']==1)).astype(int)
df['rider_driver_diff']=df['Number_of_Riders']-df['Number_of_Drivers']
df['is_preminum']=(df['Vehicle_Type']=='Premium').astype(int)
df['is_night']=(df['Time_of_Booking']=='Night').astype(int)
df['is_loyal_gold']=(df['Customer_Loyalty_Status']=='Gold').astype(int)

In [6]:
le_loc=LabelEncoder(); df['Location_enc']=le_loc.fit_transform(df['Location_Category'].astype(str))
le_loy=LabelEncoder(); df['Loyalty_enc']=le_loy.fit_transform(df['Customer_Loyalty_Status'].astype(str))
le_time=LabelEncoder(); df['Time_enc']=le_time.fit_transform(df['Time_of_Booking'].astype(str))
le_veh=LabelEncoder(); df['Vehicle_enc']=le_veh.fit_transform(df['Vehicle_Type'].astype(str))

In [8]:
TARGET= 'Historical_Cost_of_Ride'
Q1,Q3= df[TARGET].quantile(0.01), df[TARGET].quantile(0.99)
before=len(df)
df=df[(df[TARGET] >=Q1) & (df[TARGET] <=Q3)]
print(f"Outlier removal: {before-len(df)} rows. Remaining: {len(df)}")

Outlier removal: 20 rows. Remaining: 980


In [9]:
FEATURES= [
    'Number_of_Riders','Number_of_Drivers','Location_enc','Loyalty_enc',
    'Number_of_Past_Rides','Average_Ratings','Time_enc','Vehicle_enc',
    'Expected_Ride_Duration','demand_supply_ratio','high_demand_flag',
    'low_supply_flag','surge_indicator','rider_driver_diff',
    'is_premium','is_night','is_loyal_gold'
]

In [15]:
X= df[FEATURES]; y=df[TARGET]
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
scaler=StandardScaler()
X_train_sc= scaler.fit_transform(X_train)
X_test_sc= scaler.fit_transform(X_test)

In [22]:
joblib.dump(scaler,'dump/scaler.pkl')
joblib.dump(le_loc, 'dump/le_loc.pkl')
joblib.dump(le_loy, 'dump/le_loy.pkl')
joblib.dump(le_time, 'dump/le_time.pkl')
joblib.dump(le_veh, 'dump/le_veh.pkl')
df.to_csv('dump/pricepulse_processed.csv', index=False)
np.save('dump/X_train.npy', X_train_sc)
np.save('dump/X_test.npy', X_test_sc)
np.save('dump/y_train.npy', y_train.values)
np.save('dump/y_test.npy', y_test.values)
X_train.to_csv('dump/X_train_raw.csv', index=False)
X_test.to_csv('dump/X_test_raw.csv', index=False)
with open('dump/features.txt','w') as f: f.write('\n'.join(FEATURES))

In [21]:
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (784, 17) | Test: (196, 17)
